# Lecturer voice — Piper (VITS) finetune on Kaggle GPU

Fast realtime voice (non-autoregressive VITS, RTF ≪ 1). Training stack = **`OHF-Voice/piper1-gpl`**.

### ⚠️ Cell order is load-bearing — do NOT reorder
`datasets`/`pandas` need **numpy 2** (Kaggle's image); piper1-gpl pins **numpy <2**.
They cannot share one kernel. So we **pull + prep data first** (stock image), write
wavs to disk, and only **then install piper**. Everything after the install runs in
`!python` subprocesses or pure-python `huggingface_hub`, so the numpy downgrade is harmless.

### Storage: HuggingFace only
data in ← two HF datasets · model out → `HF_MODEL_REPO` · resume ↔ `resume.ckpt` there.
`/kaggle/working` is throwaway, rebuilt from HF each run.

**Before running:** Accelerator → **GPU**; Secrets → `HF_TOKEN` (*write*). Run top-to-bottom.

## 0 · CONFIG

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ['HF_TOKEN']
os.environ['HF_TOKEN'] = HF_TOKEN

HF_USER       = 'cyttic'
HF_DATASETS   = ['cyttic/audio-kri-russian', 'cyttic/audio-kri-russian-2']
HF_MODEL_REPO = f'{HF_USER}/lecturer-ru-piper'   # voice + resume.ckpt live here
RESUME_NAME   = 'resume.ckpt'
VOICE_NAME    = 'lecturer_ru'

BASE_REPO = 'rhasspy/piper-checkpoints'
BASE_DIR  = 'ru/ru_RU/irina/medium'                       # female base ~ female target
BASE_CKPT = f'{BASE_DIR}/epoch=4139-step=929464.ckpt'     # irina/medium (verified)

ESPEAK_VOICE = 'ru'
SR           = 22050        # medium quality
MAX_EPOCHS   = 6000         # ABSOLUTE; counter continues from the base ckpt (~4139)
BATCH        = 16

WORK   = '/kaggle/working'
DATA   = f'{WORK}/ds'; WAVS = f'{DATA}/wav'; META = f'{DATA}/metadata.csv'
CACHE  = f'{WORK}/cache'; CONFIG = f'{WORK}/config.json'
PRE    = f'{WORK}/pretrained'
PIPER  = f'{WORK}/piper1-gpl'

## 1 · Pull both HF datasets → `wav/` + `metadata.csv`  (stock numpy-2 env)

Runs **before** piper is installed. Raw WAV bytes decoded with `soundfile`
(`Audio(decode=False)`, no torchcodec), resampled to `SR` mono 16-bit.
CSV is piper1-gpl format: `clip.wav|text`.

In [ ]:
import io, csv, soundfile as sf, librosa
from datasets import load_dataset, Audio
os.makedirs(WAVS, exist_ok=True)
n = 0
with open(META, 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='|')
    for repo in HF_DATASETS:
        ds = load_dataset(repo, split='train', token=HF_TOKEN).cast_column('audio', Audio(decode=False))
        for r in ds:
            text = (r.get('text') or '').strip()
            if not text: continue
            a, sr = sf.read(io.BytesIO(r['audio']['bytes']), dtype='float32', always_2d=True)
            a = a.mean(axis=1)
            if sr != SR: a = librosa.resample(a, orig_sr=sr, target_sr=SR)
            n += 1
            fn = f'clip_{n:04d}.wav'
            sf.write(f'{WAVS}/{fn}', a, SR, subtype='PCM_16')
            w.writerow([fn, text])
print(f'{n} clips -> {WAVS} ({SR} Hz mono 16-bit)')
print(open(META, encoding='utf-8').read()[:300])

### 1b · Clean Whisper hallucination rows

In [ ]:
BAD = ['продолжение следует', 'субтитры', 'редактор субтитров',
       'amara.org', 'подпишись', 'спасибо за просмотр']
kept = []
for line in open(META, encoding='utf-8').read().splitlines():
    if '|' not in line: continue
    fn, text = line.split('|', 1)
    t = text.lower().strip()
    if len(t) < 3 or any(b in t for b in BAD):
        p = f'{WAVS}/{fn}'
        if os.path.exists(p): os.remove(p)
        continue
    kept.append((fn, text))
with open(META, 'w', encoding='utf-8', newline='') as f:
    csv.writer(f, delimiter='|').writerows(kept)
print(f'{len(kept)} clips after cleaning')

## 2 · Starting checkpoint — resume from HF, else base  (still stock env)

`huggingface_hub` is pure-python, so this works before *or* after the piper install.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi
os.makedirs(PRE, exist_ok=True)
api = HfApi(token=HF_TOKEN)
have = False
try:    have = RESUME_NAME in api.list_repo_files(HF_MODEL_REPO, repo_type='model')
except Exception: pass
if have:
    START_CKPT = hf_hub_download(HF_MODEL_REPO, RESUME_NAME, local_dir=PRE, token=HF_TOKEN)
    print('RESUMING from HF:', START_CKPT)
else:
    START_CKPT = hf_hub_download(BASE_REPO, BASE_CKPT, repo_type='dataset', local_dir=PRE)
    print('FIRST RUN, base:', START_CKPT)

## 3 · Install piper1-gpl  ← only now (downgrades numpy; data is already on disk)

After this cell `datasets`/`pandas` are broken in-kernel — expected. Everything below
uses `!python` subprocesses or pure-python `huggingface_hub`.

The patch at the end fixes **PyTorch ≥ 2.6**: its `torch.load` default flipped to
`weights_only=True`, which refuses the resume checkpoint (it carries a
`pathlib.PosixPath`). Without it, `fit` dies at startup and no checkpoint is written.

In [ ]:
!apt-get -qq install -y espeak-ng build-essential cmake ninja-build >/dev/null
![ -d {PIPER} ] || git clone -q https://github.com/OHF-voice/piper1-gpl.git {PIPER}
%cd {PIPER}
!pip -q install -e '.[train]' 2>&1 | tail -5
!./build_monotonic_align.sh
!python setup.py build_ext --inplace 2>&1 | tail -3
%cd /kaggle/working

# --- torch>=2.6 weights_only fix: patch piper's train entrypoint (idempotent) ---
import pathlib
_tm = pathlib.Path(PIPER) / 'src/piper/train/__main__.py'
_src = _tm.read_text()
if 'add_safe_globals' not in _src:
    _shim = '''import torch as _t, pathlib as _pl
_t.serialization.add_safe_globals([_pl.PosixPath, _pl.Path])
_orig_load = _t.load
_t.load = lambda *a, **k: _orig_load(*a, **{**k, 'weights_only': False})
'''
    _tm.write_text(_shim + _src)
print('piper.train patched for torch>=2.6:', 'add_safe_globals' in _tm.read_text())

!python -c "import piper.train; print('piper.train OK')"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 3b · Sanitize the start checkpoint — drop hyperparameters the current model rejects

Old `rhasspy/piper` base checkpoints carry model hyperparameters (e.g. `sample_bytes`)
that the newer `piper1-gpl` `VitsModel` no longer accepts. Lightning rebuilds the model
from `--ckpt_path`'s saved hyperparameters and aborts with
`fit does not accept option 'model.sample_bytes'`. We keep only keys present in
`VitsModel.__init__` and rewrite the checkpoint (weights / optimizer / epoch counter
are untouched, so resume still continues from the right epoch).

Runs in a `!python` subprocess so it uses the clean piper env (no in-kernel numpy /
`sys.path` issues). Idempotent — re-running just rewrites `start_clean.ckpt`.

In [ ]:
_san = f'{WORK}/_sanitize_ckpt.py'
pathlib.Path(_san).write_text('''
import sys, inspect, torch
from piper.train.vits.lightning import VitsModel
IN, OUT = sys.argv[1], sys.argv[2]
valid = set(inspect.signature(VitsModel.__init__).parameters) - {"self"}
ck = torch.load(IN, map_location="cpu", weights_only=False)
hp = dict(ck.get("hyper_parameters") or {})
removed = sorted(k for k in hp if k not in valid)
for k in removed:
    hp.pop(k)
ck["hyper_parameters"] = hp
torch.save(ck, OUT)
print("kept", len(hp), "model hparams; removed stale:", removed)
''')
CLEAN_CKPT = f'{PRE}/start_clean.ckpt'
!python {_san} {START_CKPT} {CLEAN_CKPT}
assert os.path.exists(CLEAN_CKPT), 'sanitize step failed — see the output above'
START_CKPT = CLEAN_CKPT
print('start checkpoint ->', START_CKPT)

## 4 · Fine-tune

`fit` folds phonemize/caching in (`--data.cache_dir`). `--ckpt_path` loads the
(sanitized) start checkpoint and **the epoch counter continues from it** (base ≈ 4139),
so `MAX_EPOCHS` is absolute. If `fit` exits instantly with no training, the base epoch
already exceeds `MAX_EPOCHS` — raise it in CONFIG and rerun.

In [ ]:
os.makedirs(CACHE, exist_ok=True)
!python -m piper.train fit \
  --data.voice_name {VOICE_NAME} \
  --data.csv_path {META} \
  --data.audio_dir {WAVS} \
  --data.espeak_voice {ESPEAK_VOICE} \
  --data.cache_dir {CACHE} \
  --data.config_path {CONFIG} \
  --data.batch_size {BATCH} \
  --model.sample_rate {SR} \
  --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.precision 32 \
  --ckpt_path {START_CKPT}
# CUDA OOM? lower --data.batch_size (8/4). Only MEDIUM base ckpts load cleanly.

## 5 · Export to ONNX (+ config → `.onnx.json`)

In [ ]:
import glob, shutil
# Lightning's default dir, plus a recursive fallback in case the version differs.
ckpts = (glob.glob(f'{WORK}/lightning_logs/*/checkpoints/*.ckpt')
         or glob.glob(f'{WORK}/**/checkpoints/*.ckpt', recursive=True))
ckpts = sorted(ckpts, key=os.path.getmtime)
assert ckpts, f'no checkpoint produced — check the fit log above (looked under {WORK})'
LAST_CKPT = ckpts[-1]; print('exporting', LAST_CKPT)
VOICE = f'{WORK}/{VOICE_NAME}.onnx'
!python -m piper.train.export_onnx --checkpoint {LAST_CKPT} --output-file {VOICE}
shutil.copy(CONFIG, VOICE + '.json')
print('voice:', VOICE, '(+ .json)')

## 6 · Save to HuggingFace (voice **and** resume checkpoint)

In [ ]:
api.create_repo(HF_MODEL_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(path_or_fileobj=VOICE,           path_in_repo=f'{VOICE_NAME}.onnx',      repo_id=HF_MODEL_REPO)
api.upload_file(path_or_fileobj=VOICE + '.json', path_in_repo=f'{VOICE_NAME}.onnx.json', repo_id=HF_MODEL_REPO)
api.upload_file(path_or_fileobj=LAST_CKPT,       path_in_repo=RESUME_NAME,               repo_id=HF_MODEL_REPO)
print('saved ->', f'https://huggingface.co/{HF_MODEL_REPO}')

## 7 · Quick listen

In [ ]:
TEXT = 'Привет! Это тест быстрого клонированного голоса лектора.'
!echo "{TEXT}" | piper -m {VOICE} -f {WORK}/test.wav
from IPython.display import Audio
Audio(f'{WORK}/test.wav')